# 04 — Transformer fine-tuning on LEDGAR (Google Colab)

Stream A (Robin), milestone **m4**. This notebook only **launches** `scripts/train_transformer.py`;
all the logic lives in `src/models/transformer.py`.

Before running:
- *Runtime → Change runtime type → GPU* (T4 / L4 / A100).
- Add a `GITHUB_TOKEN` secret (key icon in the left bar, "Notebook access" on): a GitHub token with read access
  to the private repository. It is never printed and never written to `.git/config`.

Every run writes `results/runs/<run_id>/` (config + metrics, versioned) and `results/predictions/` (gitignored)
inside the Drive copy of the repository; checkpoints go to a separate Drive folder.
**The test set is evaluated only in the final run (A6), once.**

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Clone or update the repository

The token is sent as an HTTP header for each `git` call, so the remote URL stays clean.
The Drive clone is a read-only mirror of `origin/<BRANCH>` (`main`, the only branch we work on): never edit or
commit in it.

In [ ]:
import base64
import os
import subprocess

from google.colab import userdata

REPO_URL = "https://github.com/1RLevasseurAlbertSchool/Projet_Deep_Learning.git"
REPO_DIR = "/content/drive/MyDrive/Projet_Deep_Learning"
BRANCH = "main"

_token = userdata.get("GITHUB_TOKEN")
_auth = base64.b64encode(f"x-access-token:{_token}".encode()).decode()


def git(*args):
    """Run git with the token as an HTTP header: never printed, never stored in .git/config."""
    r = subprocess.run(["git", "-c", f"http.extraHeader=Authorization: Basic {_auth}", *args],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(_token, "***").replace(_auth, "***").strip())
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed (exit code {r.returncode})")


if os.path.isdir(f"{REPO_DIR}/.git"):
    # The Drive clone is a read-only mirror: force it to origin/<BRANCH>. This discards local edits and
    # overwrites untracked files in the way (run JSONs written here by Colab, then committed from a laptop),
    # which `pull --ff-only` refuses. Other untracked runs and the gitignored predictions are kept.
    git("-C", REPO_DIR, "fetch", "origin")
    git("-C", REPO_DIR, "checkout", "-f", BRANCH)
    git("-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}")
else:
    git("clone", "--branch", BRANCH, REPO_URL, REPO_DIR)
git("-C", REPO_DIR, "log", "--oneline", "-1")

## 3. Install the dependencies

In [ ]:
%cd {REPO_DIR}
# Runtime deps only: requirements-dev.txt would upgrade the ipykernel / jupyter-client pinned by google-colab.
!pip install -q -r requirements.txt

## 4. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
print("torch", torch.__version__, "| CUDA", torch.version.cuda)

## 5. Checkpoints folder on Drive

Checkpoints are large (model + optimizer state) and gitignored: they stay on Drive, outside the repository.
`config.json` of each run records the best checkpoint path.

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/ledgar_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

## 6. Smoke test (A2)

2,000 examples per split, 1 epoch, `max_length` 128: checks that the loss goes down and that the model does not
predict a single class. A few minutes on a T4.

In [ ]:
!python scripts/train_transformer.py --model distilbert-base-uncased --variant smoke \
    --subset 2000 --epochs 1 --max-length 128 --output-dir "{CKPT_DIR}"

## 7. DistilBERT on the full training set (A4) — validation only

Defaults: `max_length` 256 (A3, see `PROGRESS.md`), 3 epochs, lr 5e-5, batch 32, 10 % warmup.

In [ ]:
!python scripts/train_transformer.py --model distilbert-base-uncased --variant full --output-dir "{CKPT_DIR}"

## 8. Other encoders (A5) — validation only, if GPU time allows

Check the token lengths with each model's own tokenizer first (the vocabularies differ).

In [ ]:
!python scripts/token_lengths.py --model nlpaueb/legal-bert-base-uncased

In [ ]:
!python scripts/train_transformer.py --model nlpaueb/legal-bert-base-uncased --variant full \
    --lr 3e-5 --batch-size 16 --output-dir "{CKPT_DIR}"

In [ ]:
!python scripts/train_transformer.py --model microsoft/deberta-v3-base --variant full \
    --lr 3e-5 --batch-size 16 --output-dir "{CKPT_DIR}"

## 9. Single test evaluation of the chosen run (A6) — no retraining

Set `FINAL_RUN` to the `run_id` of the run chosen on **validation macro-F1 only** (sections 7-8). Its best
checkpoint (on Drive, recorded in its `config.json`) predicts validation and test with the same tokenisation;
the new run `<model>_final_<timestamp>` records `source_run`. The cell refuses to run while `FINAL_RUN` is empty,
and the script refuses if the test set was already evaluated for that run. Run it **once**, then log the printed
`run_id`.

In [ ]:
FINAL_RUN = ""  # run_id chosen on validation, e.g. "legal-bert_full_20261020-1430"
assert FINAL_RUN, "Choose the final run on validation first: the test set is evaluated only once."
!python scripts/evaluate_checkpoint.py --source-run {FINAL_RUN}

## 10. Summary of the last runs

The `results/runs/` folders are committed from a machine with push access (this notebook never pushes).

In [ ]:
import json
from pathlib import Path

for path in sorted(Path("results/runs").glob("*/metrics.json"), key=lambda p: p.stat().st_mtime)[-6:]:
    m = json.loads(path.read_text())
    if "validation" not in m:
        continue
    test = f" | test macro-F1 {m['test']['macro_f1']:.4f}" if "test" in m else ""
    print(f"{path.parent.name:<40} val macro-F1 {m['validation']['macro_f1']:.4f} "
          f"| val acc {m['validation']['accuracy']:.4f}{test} | {m['train_time_s']:.0f} s")